In [11]:
# Import libraries and configure GPU
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

import tensorflow as tf

# Enable memory growth
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print('Found GPU, enabling memory growth')
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
else:
    print('GPU not found, using CPU')

# Regular imports
import pandas as pd
import gc
from typing import List, Tuple, Union
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model, load_model
from keras.callbacks import Callback
from keras.layers import Dense, LSTM, Embedding, Bidirectional, Input, Concatenate
from keras import backend as K
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


Found GPU, enabling memory growth


In [12]:
'''
DATA PREPROCESSING
1. Load data
2. Divide data into train, validation, test with the first 70%, then 15%, then 15%
3. Tokenize and pad the data
'''
filename = 'genomic_dataset_sr_train.txt'
df = pd.read_csv(filename, sep='\t', header=None, names=['1st_seq', '2nd_seq', 'label'])

In [13]:
# separate data into train, validation, test
train_data = df[:int(len(df)*0.7)]
validate_data = df[int(len(df)*0.7):int(len(df)*0.85)]
test_data = df[int(len(df)*0.85):]

# divide data into different lists
seq1_list = train_data['1st_seq'].tolist()
seq2_list = train_data['2nd_seq'].tolist()
label_list = train_data['label'].values


In [14]:
# tokenize and pad data
def data_preprocess(seq1_lst: List[str], seq2_lst: List[str]) -> Tuple[List[int], List[int], int, int]:
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(seq1_lst + seq2_lst)
    tokenized_seq1_list = tokenizer.texts_to_sequences(seq1_lst)
    tokenized_seq2_list = tokenizer.texts_to_sequences(seq2_lst)

    vocab_size = len(tokenizer.word_index) + 1
    max_len = max([len(seq) for seq in tokenized_seq1_list + tokenized_seq2_list])

    padded_seq1_list = pad_sequences(tokenized_seq1_list, maxlen=max_len, padding='post')
    padded_seq2_list = pad_sequences(tokenized_seq2_list, maxlen=max_len, padding='post')
    
    return padded_seq1_list, padded_seq2_list, vocab_size, max_len

In [15]:
padded_seq1_list, padded_seq2_list, vocab_size, max_len = data_preprocess(seq1_list, seq2_list)

In [16]:
'''
MODEL TRAINING
1. Add embedding layer
2. Add LSTM layer
3. Add Dense layer
4. Compile model
5. Fit model
6. Save/Load model
'''

#input 
X = [padded_seq1_list, padded_seq2_list]
y = label_list


In [18]:
# setup dimensions
input_dim = max_len
output_dim = 1

# setup model
input1 = Input(shape=(None,))
input2 = Input(shape=(None,))

#embedding layer
embed1 = Embedding(
    input_dim=vocab_size,
    output_dim=128,
) (input1)
embed2 = Embedding(
    input_dim=vocab_size,
    output_dim=128,
) (input2)

#concatenate layer
concat_layer = Concatenate()([embed1, embed2])

#bi-lstm layer
bi_lstm_layer = Bidirectional(
    LSTM(64, return_sequences=True)
) (concat_layer)
bi_lstm_layer = Bidirectional(
    LSTM(32)
) (bi_lstm_layer)

# Add regularization and dropout
bi_lstm_layer = Dense(32, activation='relu')(bi_lstm_layer)
bi_lstm_layer = tf.keras.layers.Dropout(0.2)(bi_lstm_layer)
output_layer = Dense(output_dim, activation='sigmoid')(bi_lstm_layer)


model = Model(inputs=[input1, input2], outputs=output_layer)
model.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

In [20]:
class CustomModelCheckpoint(Callback):
    def __init__(self, save_freq=10, filepath='model.keras'):
        super().__init__()
        self.save_freq = save_freq
        self.filepath = filepath
        self.best_val_loss = float('inf')

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.save_freq == 0:
            print(f'\nEpoch {epoch + 1}: Saving model checkpoint...')
            # Save model after training batch completes
            tf.keras.backend.set_learning_phase(0)  
            self.model.save(self.filepath, overwrite=True)
            tf.keras.backend.set_learning_phase(1)
            
        # Track best model
        if logs.get('val_loss', float('inf')) < self.best_val_loss:
            self.best_val_loss = logs.get('val_loss')
            print(f'\nEpoch {epoch + 1}: Saving best model...')
            self.model.save('model_best.keras', overwrite=True)
            
custom_checkpoint = CustomModelCheckpoint(save_freq=10) # save model every 10 epochs

In [21]:
'''
CHOOSE EITHER ONE OF THE FOLLOWING:
1. Load trained-model and continue training
2. Train new model from scratch
'''

#! 1. Load trained-model
def load_trained_model(filename: str = 'model.keras'):
    #load model from file
    model = load_model(filename)
    epochs = 10
    batch_size = 32

    # continue training
    model.fit(X, y, epochs=epochs, batch_size=batch_size, callbacks=[custom_checkpoint])

In [22]:

#! 2. TRAIN NEW MODEL
def train_new_model():
    epochs = 20
    batch_size = 2048
    # model training
    model.fit(
    X, y,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[custom_checkpoint]
)

In [ ]:
print('Start training...')
train_new_model()

Start training...
Epoch 1/20


In [7]:
'''
MODEL EVALUATION
'''
# Add after the model prediction section
def evaluate_model(dataset: pd.DataFrame, model_path: str = None, model = None) -> dict:
    """
    Evaluate model performance on given dataset
    Args:
        dataset: DataFrame containing sequences and labels
        model_path: Optional path to saved model file
        model: Optional pre-trained model instance
    """
    try:
        # Use provided model or load from file
        if model is None and model_path is not None:
            model = load_model(model_path)
        elif model is None:
            raise ValueError("Either model_path or model instance must be provided")
            
        # Process data and get predictions
        seq1 = dataset['1st_seq'].tolist()
        seq2 = dataset['2nd_seq'].tolist() 
        labels = dataset['label'].values
        
        padded_seq1, padded_seq2, _, _ = data_preprocess(seq1, seq2)
        X = [padded_seq1, padded_seq2]
        
        predictions_prob = model.predict(X)
        predictions = (predictions_prob > 0.5).astype(int)
        
        metrics = {
            'accuracy': accuracy_score(labels, predictions),
            'precision': precision_score(labels, predictions),
            'recall': recall_score(labels, predictions),
            'f1': f1_score(labels, predictions)
        }
        
        return metrics
        
    finally:
        # Only cleanup if we loaded the model
        if model_path is not None:
            del model
            gc.collect()
            K.clear_session()
            tf.keras.backend.clear_session()
            
            if tf.config.experimental.list_physical_devices('GPU'):
                for gpu in tf.config.experimental.list_physical_devices('GPU'):
                    tf.config.experimental.set_memory_growth(gpu, True)

In [8]:
# use model directly or load from file
model_path = 'model.keras' # load from saved model
datasets = {
        'TRAIN': train_data,
        'VALIDATION': validate_data,
        'TEST': test_data
    }

print('Evaluating model performance...')    
for name, dataset in datasets.items():
    metrics = evaluate_model(dataset, model=model) # use trained model directly
    # metrics = evaluate_model(dataset, model_path=model_path) # load model from file
    print(f"\n{name} SET METRICS:")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 Score:  {metrics['f1']:.4f}")


Evaluating model performance...


NameError: name 'evaluate_model' is not defined

In [10]:
'''
MODEL PREDICT
1. Load model
2. Load data
3. Predict
4. Save result
5. Analyze result
'''

def predict(filename: str='model.h5') -> List[Union[bool, float]]:
    model = load_model(filename)
    
    # sample_data = (
    #     ['test_sequence_1'], 
    #     ['test_sequence_2']
    # )
    sample_data = (
        ['test_sequence_1'],
        ['another_sequence_10']
    )
    
    processed_data = data_preprocess(sample_data[0], sample_data[1])
    X_test = list(processed_data[:2])
    
    predictions = model.predict(X_test) # output as list of matching probability
    
    
    threshold = 0.5
    bool_predictions = [True if prediction > threshold else False for prediction in predictions]
        
    return predictions
    # return bool_predictions

1/1 [==============================] - 1s 1s/step
[[0.9964434]]
1/1 [==============================] - 1s 1s/step
[[0.9964434]]


In [ ]:
print('Start predicting...')
print(predict(filename='model.keras'))

print('Done')